# AquaSynex Phase 2.5B: ML Model Experimentation & Benchmarking

**SIH26146 – AI-Powered Monitoring & Analysis of Bitcoin Transaction Traffic**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kushpagariya/AquaSynex-SIH26146/blob/ml/notebooks/05_colab_ml_experimentation.ipynb)

This notebook trains and benchmarks baseline and gradient boosted tree models on the canonical 46-feature modeling dataset (`data/processed/modeling/modeling_dataset.parquet`).

> **Anti-Leakage & Governance Protocol**:
> - Uses **strictly the 46 canonical predictive features** cataloged in `feature_manifest.yaml`.
> - Chronological split: **70% Train ($N=7,000$)**, **15% Validation ($N=1,500$)**, **15% Test ($N=1,500$)**.
> - Preprocessing (scaling and categorical encoding) is **fit strictly on the train partition**.
> - The **Test set is completely untouched and frozen** (zero model selection or tuning on test).

In [1]:
# 1. Environment Setup & Dependency Installation
!pip install -q duckdb pyarrow scikit-learn xgboost catboost pyyaml matplotlib seaborn

In [2]:
# 2. Data Loading & Repository Alignment
import os
import yaml
import duckdb
import numpy as np
import pandas as pd

# Locate dataset and feature manifest (handles local repo or Colab clone)
if os.path.exists('data/processed/modeling/modeling_dataset.parquet'):
    pq_path = 'data/processed/modeling/modeling_dataset.parquet'
    manifest_path = 'data/processed/modeling/feature_manifest.yaml'
elif os.path.exists('../data/processed/modeling/modeling_dataset.parquet'):
    pq_path = '../data/processed/modeling/modeling_dataset.parquet'
    manifest_path = '../data/processed/modeling/feature_manifest.yaml'
else:
    # In Colab, clone repo if not already cloned
    !git clone https://github.com/kushpagariya/AquaSynex-SIH26146.git
    pq_path = 'AquaSynex-SIH26146/data/processed/modeling/modeling_dataset.parquet'
    manifest_path = 'AquaSynex-SIH26146/data/processed/modeling/feature_manifest.yaml'

with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = yaml.safe_load(f)

con = duckdb.connect()
df_raw = con.execute(f"SELECT * FROM read_parquet('{pq_path}')").df()
con.close()

canonical_features = manifest['canonical_feature_names']
print(f"[*] Loaded Modeling Dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"[*] Canonical Feature Space: {len(canonical_features)} features (44 numeric + 2 categorical)")

[*] Loaded Modeling Dataset: 10,000 rows x 56 columns
[*] Canonical Feature Space: 46 features (44 numeric + 2 categorical)


In [3]:
# 3. Chronological Temporal Partitioning
train_mask = df_raw['temporal_split'] == 'train'
val_mask = df_raw['temporal_split'] == 'val'
test_mask = df_raw['temporal_split'] == 'test'

df_train = df_raw[train_mask].copy()
df_val = df_raw[val_mask].copy()
df_test = df_raw[test_mask].copy()

y_train = df_train['target_binary'].values
y_val = df_val['target_binary'].values
# Note: y_test is NOT accessed to preserve out-of-time test integrity

print(f"[*] Train split: {len(df_train):,} rows ({len(df_train)/len(df_raw):.1%}) | Suspicious: {df_train['target_binary'].mean():.2%}")
print(f"[*] Val split:   {len(df_val):,} rows ({len(df_val)/len(df_raw):.1%}) | Suspicious: {df_val['target_binary'].mean():.2%}")
print(f"[*] Test split:  {len(df_test):,} rows ({len(df_test)/len(df_raw):.1%}) | Suspicious: {df_test['target_binary'].mean():.2%} (STRICTLY FROZEN)")

[*] Train split: 7,000 rows (70.0%) | Suspicious: 44.91%
[*] Val split:   1,500 rows (15.0%) | Suspicious: 42.13%
[*] Test split:  1,500 rows (15.0%) | Suspicious: 37.40% (STRICTLY FROZEN)


In [4]:
# 4. Preprocessing Fitted Strictly on Train Split
from sklearn.preprocessing import RobustScaler, OneHotEncoder

cat_cols = ['net_country', 'net_asn']
num_cols = [c for c in canonical_features if c not in cat_cols]

scaler = RobustScaler()
scaler.fit(df_train[num_cols])

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_train[cat_cols].astype(str))

ohe_feature_names = list(ohe.get_feature_names_out(cat_cols))
processed_feature_names = num_cols + ohe_feature_names

# Transform train and validation sets
X_train_num = scaler.transform(df_train[num_cols])
X_train_cat = ohe.transform(df_train[cat_cols].astype(str))
X_train = np.hstack([X_train_num, X_train_cat])

X_val_num = scaler.transform(df_val[num_cols])
X_val_cat = ohe.transform(df_val[cat_cols].astype(str))
X_val = np.hstack([X_val_num, X_val_cat])

print(f"[+] Preprocessing Complete: {len(num_cols)} numeric + {len(cat_cols)} categorical -> {X_train.shape[1]} input dimensions.")

[+] Preprocessing Complete: 44 numeric + 2 categorical -> 71 input dimensions.


In [5]:
# 5. Model Training & Evaluation Suite
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

models = {
    'Baseline (Logistic Regression)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss', n_jobs=-1),
    'CatBoost': CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, random_seed=42, verbose=0)
}

results = []
trained_models = {}
val_predictions = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    t_fit = time.time() - t0
    trained_models[name] = model

    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else preds
    val_predictions[name] = (preds, probs)

    train_preds = model.predict(X_train)
    train_probs = model.predict_proba(X_train)[:, 1] if hasattr(model, 'predict_proba') else train_preds

    results.append({
        'Model': name,
        'Val ROC-AUC': roc_auc_score(y_val, probs),
        'Val PR-AUC': average_precision_score(y_val, probs),
        'Val F1-Score': f1_score(y_val, preds),
        'Val Recall': recall_score(y_val, preds),
        'Val Precision': precision_score(y_val, preds),
        'Val Accuracy': accuracy_score(y_val, preds),
        'Train F1': f1_score(y_train, train_preds),
        'F1 Gap (Train-Val)': f1_score(y_train, train_preds) - f1_score(y_val, preds),
        'Fit Time (s)': round(t_fit, 3)
    })

df_benchmark = pd.DataFrame(results)
print('=== MODEL BENCHMARK RESULTS (EVALUATED ON VALIDATION N=1,500) ===')
print(df_benchmark.round(4).to_string(index=False))

=== MODEL BENCHMARK RESULTS (EVALUATED ON VALIDATION N=1,500) ===
                         Model  Val ROC-AUC  Val PR-AUC  Val F1-Score  Val Recall  Val Precision  Val Accuracy  Train F1  F1 Gap (Train-Val)  Fit Time (s)
Baseline (Logistic Regression)       0.6122      0.6598        0.4977      0.5965          0.427        0.4927    0.4895             -0.0082         0.116
                 Random Forest       0.9999      0.9998        0.9944      0.9889          1.000        0.9953    0.9958              0.0014         0.190
                       XGBoost       1.0000      1.0000        0.9992      0.9984          1.000        0.9993    0.9997              0.0005         1.810
                      CatBoost       1.0000      1.0000        0.9992      0.9984          1.000        0.9993    0.9995              0.0003         0.617


In [6]:
# 6. Confusion Matrices Inspection
print('=== CONFUSION MATRICES (VALIDATION SPLIT) ===')
for name in models.keys():
    preds, _ = val_predictions[name]
    cm = confusion_matrix(y_val, preds)
    print(f'\nModel: {name}')
    print(f'  True Negatives:  {cm[0, 0]:>4} | False Positives: {cm[0, 1]:>4}')
    print(f'  False Negatives: {cm[1, 0]:>4} | True Positives:  {cm[1, 1]:>4}')

=== CONFUSION MATRICES (VALIDATION SPLIT) ===

Model: Baseline (Logistic Regression)
  True Negatives:   362 | False Positives:  506
  False Negatives:  255 | True Positives:   377

Model: Random Forest
  True Negatives:   868 | False Positives:    0
  False Negatives:    7 | True Positives:   625

Model: XGBoost
  True Negatives:   868 | False Positives:    0
  False Negatives:    1 | True Positives:   631

Model: CatBoost
  True Negatives:   868 | False Positives:    0
  False Negatives:    1 | True Positives:   631


In [7]:
# 7. Feature Importance Analysis (Tree Models)
print('=== TOP 10 FEATURE IMPORTANCES (CatBoost) ===')
cb_model = trained_models['CatBoost']
importances = cb_model.get_feature_importance()
df_imp = pd.DataFrame({
    'feature': processed_feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(df_imp.head(10).round(3).to_string(index=False))

print('\n[+] Phase 2.5B Experimentation Complete.')
print('[+] Test set remains completely untouched and frozen.')

=== TOP 10 FEATURE IMPORTANCES (CatBoost) ===
                      feature  importance
       rel_change_value_ratio      53.656
 net_is_standard_bitcoin_port       9.833
hist_out_mean_neighbor_degree       9.796
                 net_src_port       5.613
time_since_prev_global_tx_sec       4.319
             time_day_of_week       1.899
             time_txs_last_1m       1.479
              tx_output_count       1.155
     tx_fee_rate_sat_per_byte       1.084
                   tx_log_fee       1.021

[+] Phase 2.5B Experimentation Complete.
[+] Test set remains completely untouched and frozen.
